# 🧠 Predictive Workforce Wellbeing & Burnout Risk Analytics
### End-to-End Machine Learning Pipeline: Data Ingestion, Feature Engineering, Multi-Task Modeling, Model Calibration & Explainability (SHAP)

**Author:** Shivanshu Rai  
**Framework:** Scikit-Learn, XGBoost, SHAP, Pandas  
**Dataset:** Enterprise Workforce Dataset ($N = 178,750$ employee records)  


## 1. Environment Setup & Library Imports

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, roc_auc_score, f1_score, roc_curve, precision_recall_curve

# Set styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'DejaVu Sans'
print("All libraries successfully imported.")

## 2. Data Ingestion & Dataset Overview

In [ ]:
# Load master dataset
data_path = "artifacts/master_workforce_dataset.csv"
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("Sample Data:")
display(df.head())

print("\nDataset Info:")
df.info()

## 3. Composite Feature Engineering
Formulating domain-specific composite indexes:
- **Workload Index**: Weighted synthesis of work hours, overtime, meetings, and deadlines missed.
- **Wellness Score**: Composite metric of sleep, exercise, work-life balance, and job satisfaction.
- **Support Index**: Manager support metric scaled to $[0, 100]$.
- **Meeting Load Ratio**: Meeting count relative to standard 3 meetings per day.

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    df_eng = data.copy()
    
    # 1. Workload Index
    hours_norm = df_eng["work_hours_per_week"] / 40.0
    overtime_norm = df_eng["overtime_hours"] / 10.0
    meetings_norm = df_eng["meetings_per_day"] / 5.0
    deadlines_norm = df_eng["deadlines_missed"] / 3.0
    
    df_eng["workload_index"] = (
        hours_norm * 0.4 +
        overtime_norm * 0.3 +
        meetings_norm * 0.2 +
        deadlines_norm * 0.1
    ) * 100.0
    
    # 2. Wellness Score
    sleep_norm = (df_eng["sleep_hours"] / 8.0).clip(0.0, 1.2)
    activity_norm = df_eng["physical_activity_days"] / 7.0
    balance_norm = df_eng["work_life_balance"] / 10.0
    satisfaction_norm = df_eng["job_satisfaction"] / 10.0
    
    df_eng["wellness_score"] = (
        sleep_norm * 0.3 +
        activity_norm * 0.2 +
        balance_norm * 0.3 +
        satisfaction_norm * 0.2
    ) * 100.0
    
    # 3. Support Index & Meeting Load
    df_eng["support_index"] = df_eng["manager_support"] * 10.0
    df_eng["meeting_load"] = df_eng["meetings_per_day"] / 3.0
    
    return df_eng

df_engineered = engineer_features(df)
print("Engineered features added successfully:")
display(df_engineered[['workload_index', 'wellness_score', 'support_index', 'meeting_load']].head())

## 4. Exploratory Data Analysis & Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Burnout Score Distribution
sns.histplot(df["burnout_score"], kde=True, color="#1a6fc4", ax=axes[0])
axes[0].axvline(4.5, color="red", linestyle="--", label="Threshold (tau = 4.5)")
axes[0].set_title("Target Distribution: Burnout Score (0-10)")
axes[0].legend()

# Work Hours vs Burnout Score
sns.scatterplot(data=df.sample(2000, random_state=42), x="work_hours_per_week", y="burnout_score", hue="work_mode", alpha=0.6, ax=axes[1])
axes[1].set_title("Work Hours per Week vs. Burnout Score")

# Burnout by Job Role
role_order = df.groupby("job_role")["burnout_score"].mean().sort_values(ascending=False).index
sns.barplot(data=df, y="job_role", x="burnout_score", order=role_order, color="#0f2d56", ax=axes[2])
axes[2].set_title("Average Burnout Score by Job Role")

plt.tight_layout()
plt.show()

## 5. Preprocessing & Train/Test Split

In [ ]:
# Feature selection
features = [
    "caffeine_intake", "company_size", "deadlines_missed", "experience_years", "job_role",
    "job_satisfaction", "manager_support", "meetings_per_day", "overtime_hours",
    "physical_activity_days", "screen_time_hours", "sleep_hours", "social_support_score",
    "stress_level", "work_hours_per_week", "work_life_balance", "work_mode",
    "workload_index", "wellness_score", "support_index", "meeting_load"
]
target_score = "burnout_score"
threshold = 4.5

X = df_engineered[features]
y_score = df_engineered[target_score]
y_risk = (y_score >= threshold).astype(int)

# Train / Test Split (80% Train, 20% Test)
X_train, X_test, y_train_score, y_test_score, y_train_risk, y_test_risk = train_test_split(
    X, y_score, y_risk, test_size=0.20, random_state=42, stratify=y_risk
)

print(f"Training Set: {X_train.shape[0]:,} samples | Testing Set: {X_test.shape[0]:,} samples")

## 6. Task A: Continuous Burnout Score Regression ($y \in [0, 10]$)

In [ ]:
numeric_cols = [c for c in features if c not in {"company_size", "job_role", "work_mode"}]
categorical_cols = ["company_size", "job_role", "work_mode"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# Build Regressor Pipeline
regressor_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=8, min_samples_leaf=20, random_state=42))
])

print("Training HistGradientBoostingRegressor...")
regressor_pipeline.fit(X_train, y_train_score)

# Evaluation
y_pred_score = np.clip(regressor_pipeline.predict(X_test), 0, 10)
mae = mean_absolute_error(y_test_score, y_pred_score)
rmse = np.sqrt(mean_squared_error(y_test_score, y_pred_score))
r2 = r2_score(y_test_score, y_pred_score)

print(f"\n--- Task A Regression Metrics ---")
print(f"Mean Absolute Error (MAE) : {mae:.4f}")
print(f"Root Mean Squared Error    : {rmse:.4f}")
print(f"R² Score                   : {r2:.4f}")

## 7. Task B: Calibrated Elevated Risk Classification (\tau = 4.5)

In [ ]:
base_classifier = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=8, min_samples_leaf=20, random_state=42))
])

# Calibrate probabilities using Sigmoid calibration
calibrated_classifier = CalibratedClassifierCV(estimator=base_classifier, method="sigmoid", cv=3)

print("Training CalibratedClassifierCV...")
calibrated_classifier.fit(X_train, y_train_risk)

# Predict probabilities and classes
y_pred_prob = calibrated_classifier.predict_proba(X_test)[:, 1]
y_pred_class = (y_pred_prob >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test_risk, y_pred_prob)
f1 = f1_score(y_test_risk, y_pred_class)

print(f"\n--- Task B Classification Metrics ---")
print(f"ROC-AUC Score : {roc_auc:.4f}")
print(f"F1-Score      : {f1:.4f}\n")
print("Classification Report:")
print(classification_report(y_test_risk, y_pred_class, target_names=["Standard Risk", "Elevated Risk"]))

## 8. ROC & Precision-Recall Curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test_risk, y_pred_prob)
precision, recall, _ = precision_recall_curve(y_test_risk, y_pred_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color="#1a6fc4", lw=2, label=f"ROC Curve (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[0].set_title("Receiver Operating Characteristic (ROC) Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

# Precision-Recall Curve
axes[1].plot(recall, precision, color="#b95f00", lw=2, label="PR Curve")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Interactive Prediction Demo Function

In [ ]:
def predict_employee_burnout(employee_dict: dict):
    """Predict continuous burnout score and elevated risk probability for a single employee."""
    # Get defaults from training median
    defaults = X.median(numeric_only=True).to_dict()
    defaults['company_size'] = X['company_size'].mode()[0]
    defaults['job_role'] = X['job_role'].mode()[0]
    defaults['work_mode'] = X['work_mode'].mode()[0]
    
    # Override with input
    for k, v in employee_dict.items():
        if k in defaults:
            defaults[k] = v
            
    row_df = pd.DataFrame([defaults])
    row_df = engineer_features(row_df)
    
    score = float(regressor_pipeline.predict(row_df)[0])
    prob = float(calibrated_classifier.predict_proba(row_df)[0, 1]) * 100.0
    band = "Elevated" if score >= threshold else ("Moderate" if score >= 2.5 else "Low")
    
    print("=" * 50)
    print(f"🎯 PREDICTED BURNOUT ASSESSMENT")
    print("=" * 50)
    print(f"Burnout Score (0-10)     : {score:.2f} / 10")
    print(f"Elevated Risk Probability : {prob:.1f}%")
    print(f"Risk Band                : {band}")
    print("=" * 50)

# Sample Demo Run
sample_employee = {
    "stress_level": 8.5,
    "work_hours_per_week": 55,
    "overtime_hours": 12,
    "meetings_per_day": 6,
    "work_life_balance": 3.0,
    "manager_support": 4.0,
    "job_role": "Software Engineer",
    "work_mode": "Hybrid"
}
predict_employee_burnout(sample_employee)

## 10. Save Model Artifacts

In [ ]:
os.makedirs("artifacts", exist_ok=True)
joblib.dump(regressor_pipeline, "artifacts/burnout_score_model.joblib")
joblib.dump(calibrated_classifier, "artifacts/elevated_risk_model.joblib")

metrics = {
    "regressor": {"mae": round(mae, 4), "rmse": round(rmse, 4), "r2": round(r2, 4)},
    "risk_model": {"threshold": threshold, "roc_auc": round(roc_auc, 4), "f1": round(f1, 4)}
}
with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("All model artifacts successfully exported to artifacts/ folder.")